# Telegram Sticker Booth, step by step

This notebook turns **one photo into one Telegram sticker**, the same way the command-line tools do, but one stage at a time so you can see what each step does.

The idea comes from Google's **Sticker Kaki**: give an image model your photo plus a carefully worded prompt, have it draw a sticker on a flat green screen, then cut the sticker out and package it for Telegram.

```
photo + prompt ──► Gemini image edit ──► green-screen sticker ──► chroma key + white border ──► 512 px PNG/WEBP ──► Telegram pack
```

**Runs free by default.** `USE_API = False` skips the paid Gemini call and loads a sample generation from `examples/sample_raw.png`, so every cell below still runs top to bottom. Set it to `True` (and put `GEMINI_API_KEY` in `.env`) to draw a sticker from your own photo, at about US$0.034 per image.

In [ ]:
# Settings: the only cell you need to edit
USE_API = False          # True = call Gemini with your photo (costs ~$0.034); False = use the sample raw image
UPLOAD  = False          # True = really create a Telegram pack (needs TELEGRAM_BOT_TOKEN and TELEGRAM_USER_ID)

PHOTO = "photos/me.jpg"  # your photo, relative to the repo root (photos/ is git-ignored)
STYLE = "pixel"          # cartoon | cyberpunk | pixel  (pixel = Sticker Kaki "8-Bit Retro")
EMOJI = "😂"             # any key of EMOJIS in sticker_booth/prompts.py

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
from PIL import Image
from IPython.display import display

from sticker_booth import postprocess as pp
from sticker_booth.env import load_env
from sticker_booth.prompts import EMOJIS, STYLES, build_prompt

load_env(ROOT / ".env")  # reads keys into the environment; nothing is printed
OUT = ROOT / "notebook" / "out"
OUT.mkdir(parents=True, exist_ok=True)

def checker(size, n=16):
    """Grey checkerboard, the usual way to show transparency."""
    y, x = np.indices(size[::-1]) // n
    g = np.where((x + y) % 2, 200, 235).astype(np.uint8)
    return Image.fromarray(np.dstack([g, g, g, np.full_like(g, 255)]), "RGBA")

def show(*images, height=300, gap=12):
    """Show images side by side at the same height; transparent areas appear as a checkerboard."""
    tiles = []
    for im in images:
        im = im.convert("RGBA")
        im = im.resize((round(im.width * height / im.height), height), Image.LANCZOS)
        bg = checker(im.size); bg.alpha_composite(im); tiles.append(bg)
    sheet = Image.new("RGB", (sum(t.width for t in tiles) + gap * (len(tiles) - 1), height), "white")
    x = 0
    for t in tiles:
        sheet.paste(t.convert("RGB"), (x, 0)); x += t.width + gap
    display(sheet)

name = EMOJIS[EMOJI][0]
print(f"Making the {EMOJI} '{name}' sticker in style '{STYLE}'  (USE_API={USE_API})")

## 1. The prompt

Every sticker uses the same formula, borrowed from Sticker Kaki:

* a **scene** for the emoji ("laughing so hard that tears of joy fly off their face"),
* a **likeness** clause that lists the facial features to keep, so the sticker still looks like you,
* the **style** guidance,
* **visual details**: a colour palette for props only, a rule to keep your *real clothes and colours*, and a thick white die-cut border on a flat `#00FF00` background.

That last part matters most for automation. A flat, known green is what lets the next steps cut the sticker out without any AI background removal.

In [ ]:
prompt = build_prompt(EMOJI, STYLE)
print(prompt.replace(". ", ".\n"))

## 2. Ask Gemini to draw it

The photo is resized to 1024 px, sent as JPEG together with the prompt to `gemini-3.1-flash-lite-image`, and the reply contains the generated image.

Two errors you may hit:

* **429** with "limit: 0": the key is on the free tier, which allows no image generations through the API. Enable billing.
* **402** "prepayment credits are depleted": top up at [ai.studio/projects](https://ai.studio/projects).

With `USE_API = False` this cell loads a generation made earlier instead.

In [ ]:
raw_path = OUT / f"raw_{name}.png"
if USE_API:
    from sticker_booth.generate import gemini, photo_jpeg
    photo = ROOT / PHOTO
    raw_path.write_bytes(gemini(photo_jpeg(photo), prompt))
    show(Image.open(photo), Image.open(raw_path))
else:
    raw_path = ROOT / "examples" / "sample_raw.png"

raw = Image.open(raw_path).convert("RGB")
print("raw generation:", raw.size)
show(raw)

## 3. Cut it out: chroma key

For every pixel, "greenness" is `G - max(R, B)`. Pure `#00FF00` scores 255, while skin, hair and a white border score close to 0. Instead of a hard cut-off, two thresholds (`lo=40`, `hi=170`) give a soft ramp, so anti-aliased edges keep partial transparency.

Image models never paint a perfectly flat green, so a soft key is much more forgiving than a single threshold.

In [ ]:
rgb = np.asarray(raw).astype(np.float32)
alpha_key = pp.chroma_key(rgb)                      # 1 = keep, 0 = background
show(raw, Image.fromarray((alpha_key * 255).astype(np.uint8)))

## 4. Keep one sticker, fill its holes, remove the green fringe

* **Largest blob only.** Stray specks, floating confetti outside the border and the little Gemini sparkle watermark in the corner are dropped.
* **Fill holes.** Everything inside the outer outline becomes solid, so a green shirt or a green prop inside the white border is not punched out.
* **Despill.** Edge pixels are a mix of sticker and green. The mix is reversed (`C = a·F + (1−a)·K`, solve for `F`) and leftover green is clamped, so there is no green halo.

In [ ]:
alpha = pp.keep_largest(alpha_key)
clean = pp.clean_edges(rgb, alpha)
cutout = np.dstack([clean, alpha * 255]).astype(np.uint8)
cutout[alpha == 0, :3] = 0
show(Image.fromarray(cutout, "RGBA"))

## 5. Fit to 512 px and add the white border

Telegram static stickers must have one side of exactly **512 px**. The cut-out is cropped to its bounding box, scaled evenly (never squashed) to fit, centred on a 512×512 transparent canvas, and then given a 10 px white stroke so it reads well on both light and dark chat backgrounds.

`pp.process()` runs steps 3 to 5 in one call. It is what the CLI uses.

In [ ]:
sticker = pp.process(raw_path)          # same as steps 3-5 above
png, webp = pp.save_sticker(sticker, OUT / name)
print(f"{png.name}: {sticker.size} {sticker.mode}, {png.stat().st_size // 1024} KB")
print(f"{webp.name}: {webp.stat().st_size // 1024} KB (Telegram limit 512 KB)")
assert sticker.size == (512, 512) and sticker.mode == "RGBA"
show(raw, sticker)

## 6. A whole pack: contact sheet

One sticker is nice; a pack is the point. The CLI loops over every emoji in `prompts.py` (and over every photo in a folder), writes a `pack.json` that maps each file to its emoji, and draws a contact sheet so you can check them all at a glance:

```bash
python -m sticker_booth.generate photos/me.jpg --style pixel --out out
```

Here is the 8-Bit pack that command produced.

In [ ]:
from sticker_booth.prompts import NAMES
ex = ROOT / "examples" / "stickers"
items = [(f"{NAMES[p.stem]} {p.stem}", Image.open(p)) for p in sorted(ex.glob("*.png"))]
display(pp.contact_sheet(items, cols=4, tile=200))

## 7. Upload to Telegram (optional)

Telegram's official **@Stickers** bot only works by hand, one sticker per message. Your *own* bot can do the same thing through the Bot API (`uploadStickerFile`, `createNewStickerSet`, `addStickerToSet`), and the pack still belongs to you:

1. In Telegram, message **@BotFather**, send `/newbot`, and put the token in `.env` as `TELEGRAM_BOT_TOKEN`.
2. Open a chat with your bot and press **Start**. A bot can only create packs for users who have started it.
3. Find your user id with `python -m sticker_booth.telegram_pack --whoami` and put it in `.env` as `TELEGRAM_USER_ID`.

Pack names must end in `_by_<botusername>`. The link is `https://t.me/addstickers/<name>`. With `UPLOAD = False` this cell only does a dry run.

In [ ]:
import os
from sticker_booth.telegram_pack import build_pack

link = build_pack([ex], name="my_8bit", title="My 8-Bit",
                  user_id=os.environ.get("TELEGRAM_USER_ID"),
                  chat_id=os.environ.get("TELEGRAM_USER_ID"),   # DM the link to yourself
                  dry_run=not UPLOAD)
print(link)

## Wrap-up

| step | what | where |
|---|---|---|
| prompt | Sticker Kaki formula: scene + likeness + style + green screen | `sticker_booth/prompts.py` |
| draw | Gemini image edit, about $0.034 per sticker | `sticker_booth/generate.py` |
| cut out | soft chroma key, largest blob, despill, 512 px, white stroke | `sticker_booth/postprocess.py` |
| publish | your own bot builds the pack and sends you the link | `sticker_booth/telegram_pack.py` |

Only make stickers of people who have said yes, and delete their photos if they ask.